# TP 01 — Tool Use & AI Agents
### Version simplifiee

Trois outils Python, une boucle tool use ecrite a la main, puis le meme comportement
reconstruit en LangGraph.

| Partie | Fichier | `pytest -q` affiche |
|---|---|---|
| Depart | — | 20 failed, 8 passed |
| Partie 1 | `agent/outils.py` | 13 passed (sur `test_outils.py`) |
| Partie 2 -- Tool Use | `agent/boucle.py` | 20 passed (sur `test_outils.py` + `test_boucle.py`) |
| Partie 3 -- Agent | `agent/graphe.py` | 28 passed (suite complete) |

La partie 2 utilise directement l'API (le "tool use" brut). La partie 3 reconstruit le meme
comportement avec LangGraph, un framework d'agent.

Chaque fichier contient un exemple deja ecrit (une fonction ou un noeud complet), qui montre
le patron a suivre pour le reste.

**Cas d'usage** : un portail de sinistres pour un assureur automobile (OGI Assurance). L'agent
repond aux courtiers en consultant des polices, des clauses de garantie, et en calculant des
indemnites — sans jamais inventer un montant ou une couverture.


---
# Mise en route

**Prerequis**
- Google Colab (ou Python 3.10+ en local).
- Une cle API Anthropic dans les secrets Colab, nommee `ANTHROPIC_API_KEY` (icone cle a gauche),
  utile uniquement pour la cellule optionnelle d'essai contre le vrai Claude.
- Les 28 tests fonctionnent **sans aucun appel API** grace au double de modele deterministe fourni
  dans `tests/doubles.py`.


In [ ]:
!pip install -q pytest anthropic langgraph langchain-anthropic langchain-core typing_extensions

import os

os.makedirs("portail-agent/agent", exist_ok=True)
os.makedirs("portail-agent/tests", exist_ok=True)
os.chdir("portail-agent")
print("Repertoire de travail :", os.getcwd())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 34.4 MB/s eta 0:00:00
Repertoire de travail : /content/portail-agent


In [ ]:
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except (ImportError, Exception):
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
    print("Cle API detectee. La cellule d'appel reel pourra s'executer.")
else:
    print("Aucune cle API trouvee. Les 28 tests fonctionneront quand meme (double), "
          "seule la cellule 'Appel API reel' sera a sauter.")


Cle API detectee. La cellule d'appel reel pourra s'executer.


### `pytest.ini` et paquets Python

`agent/` et `tests/` doivent etre des paquets Python (`__init__.py`) pour que `from agent.outils import ...` et `from tests.doubles import ...` fonctionnent. `pytest.ini` a la racine evite les erreurs de collection.


In [ ]:
%%writefile pytest.ini
[pytest]
pythonpath = .
testpaths = tests


Writing pytest.ini


In [ ]:
%%writefile agent/__init__.py
# marque agent/ comme paquet Python


Writing agent/__init__.py


In [ ]:
%%writefile tests/__init__.py
# marque tests/ comme paquet Python


Writing tests/__init__.py


---
# Fichiers fournis : double et tests

`tests/doubles.py` simule Claude de facon deterministe, pour la boucle a la main et pour
LangGraph — aucun appel reseau. Les trois fichiers `test_*.py` contiennent les 28 tests qui
definissent la specification du TP. Ces fichiers ne sont **jamais modifies**.


In [ ]:
%%writefile tests/doubles.py
"""Doubles de test - fournis, ne pas modifier.

Simulent Claude de facon deterministe, pour la boucle a la main
(API brute) et pour LangGraph (messages LangChain).
"""
import re


# ═══════════ Double pour l'API brute (bloc 1) ═══════════
class _Bloc:
    def __init__(self, **kw):
        self.__dict__.update(kw)


class _Rep:
    def __init__(self, stop_reason, content):
        self.stop_reason = stop_reason
        self.content = content


class ClientBrutDouble:
    """Rejoue un scenario tool use realiste sur l'API brute."""

    def __init__(self):
        self.tour = 0

    class _M:
        def __init__(self, outer):
            self.outer = outer

        def create(self, model=None, max_tokens=None, system=None,
                   tools=None, messages=None, **kw):
            self.outer.tour += 1
            humain = " ".join(
                m["content"] for m in messages
                if m["role"] == "user" and isinstance(m["content"], str)
            ).lower()
            deja = []
            for m in messages:
                if m["role"] == "assistant" and not isinstance(m["content"], str):
                    deja += [b.name for b in m["content"]
                             if getattr(b, "type", "") == "tool_use"]
            erreurs = any(
                bloc.get("is_error")
                for m in messages
                if m["role"] == "user" and isinstance(m["content"], list)
                for bloc in m["content"])

            if erreurs:
                return _Rep("end_turn", [_Bloc(
                    type="text",
                    text="Le numero de police est introuvable. "
                         "Pouvez-vous le verifier ?")])

            if "chercher_police" not in deja:
                num = re.search(r"pol-\w{4}", humain)
                num = num.group(0).upper() if num else "POL-0000"
                return _Rep("tool_use", [
                    _Bloc(type="text", text="Je consulte la police."),
                    _Bloc(type="tool_use", id="t1", name="chercher_police",
                          input={"numero": num})])

            if "rechercher_clause" not in deja:
                return _Rep("tool_use", [
                    _Bloc(type="tool_use", id="t2",
                          name="rechercher_clause",
                          input={"sujet": "refoulement"})])

            if "calculer_indemnite" not in deja:
                return _Rep("tool_use", [
                    _Bloc(type="tool_use", id="t3",
                          name="calculer_indemnite",
                          input={"montant_dommages": 18000,
                                 "franchise": 1000, "maximum": 25000})])

            return _Rep("end_turn", [_Bloc(
                type="text",
                text="Indemnite : 17000 $ apres franchise de 1000 $ "
                     "(clause 7.3.2, p.12).")])

    @property
    def messages(self):
        return ClientBrutDouble._M(self)


class ClientBrutBoucle:
    """Redemande indefiniment le meme outil : teste le garde-fou."""

    class _M:
        def create(self, **kw):
            return _Rep("tool_use", [
                _Bloc(type="tool_use", id="x", name="rechercher_clause",
                      input={"sujet": "collision"})])

    @property
    def messages(self):
        return ClientBrutBoucle._M()


# ═══════════ Double pour LangGraph (blocs 3 a 6) ═══════════
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage


class ModeleDouble:
    """Mime llm.bind_tools(...).invoke(messages)."""

    def bind_tools(self, outils):
        return self

    def invoke(self, messages, **kw):
        deja = [m.name for m in messages if isinstance(m, ToolMessage)]
        humain = " ".join(
            m.content for m in messages
            if isinstance(m, HumanMessage)).lower()
        erreur = any("introuvable" in str(m.content).lower()
                     for m in messages if isinstance(m, ToolMessage))

        if erreur:
            return AIMessage(content="Le numero de police est introuvable. "
                                     "Pouvez-vous le verifier ?")
        if "chercher_police" not in deja:
            num = re.search(r"pol-\w{4}", humain)
            num = num.group(0).upper() if num else "POL-0000"
            return AIMessage(content="", tool_calls=[{
                "name": "chercher_police", "args": {"numero": num},
                "id": "c1"}])
        if "rechercher_clause" not in deja:
            return AIMessage(content="", tool_calls=[{
                "name": "rechercher_clause",
                "args": {"sujet": "refoulement"}, "id": "c2"}])
        if "calculer_indemnite" not in deja:
            return AIMessage(content="", tool_calls=[{
                "name": "calculer_indemnite",
                "args": {"montant_dommages": 18000, "franchise": 1000,
                         "maximum": 25000}, "id": "c3"}])
        return AIMessage(content="Indemnite de 17000 $ apres franchise de "
                                 "1000 $ (clause 7.3.2, p.12).")


class ModeleBoucle:
    """Redemande indefiniment le meme outil."""

    def bind_tools(self, o):
        return self

    def invoke(self, messages, **kw):
        return AIMessage(content="", tool_calls=[{
            "name": "rechercher_clause", "args": {"sujet": "collision"},
            "id": "x"}])


Writing tests/doubles.py


In [ ]:
%%writefile tests/test_outils.py
"""Partie 1 - les outils. Tests FOURNIS, ne pas modifier.

Version allegee (seance 1h30) : les cas limites redondants ont ete
retires, seuls les comportements essentiels restent verifies.
"""
import pytest

from agent.outils import (
    BASE_DOSSIERS, CLAUSES, DISPATCH, OUTILS_SCHEMA, SYSTEM,
    chercher_police, rechercher_clause, calculer_indemnite,
)


# ── chercher_police ──
def test_police_existante():
    assert chercher_police("POL-4471")["avenants"] == ["RE-04"]


def test_police_inconnue_message_actionnable():
    with pytest.raises(KeyError) as e:
        chercher_police("POL-9999")
    assert "verifiez" in str(e.value).lower()


# ── rechercher_clause ──
def test_clause_refoulement_mentionne_l_avenant():
    assert "RE-04" in rechercher_clause("refoulement")


def test_clause_cite_la_source():
    assert "p.12" in rechercher_clause("collision")


def test_sujet_inconnu_liste_les_options():
    r = rechercher_clause("incendie")
    assert "Aucune clause" in r and "refoulement" in r


# ── calculer_indemnite ──
def test_indemnite_simple():
    r = calculer_indemnite(18000, 1000, 25000)
    assert r["indemnite"] == 17000.0 and r["plafonne"] is False


def test_indemnite_plafonnee():
    r = calculer_indemnite(40000, 1000, 25000)
    assert r["indemnite"] == 25000.0 and r["plafonne"] is True


def test_indemnite_jamais_negative():
    assert calculer_indemnite(300, 500)["indemnite"] == 0.0


# ── schemas JSON ──
def test_trois_outils_declares():
    assert len(OUTILS_SCHEMA) == 3


def test_les_noms_correspondent_au_dispatch():
    noms = {o["name"] for o in OUTILS_SCHEMA}
    assert noms == set(DISPATCH)


def test_chaque_outil_a_les_cles_requises():
    for o in OUTILS_SCHEMA:
        assert set(o) == {"name", "description", "input_schema"}
        assert o["input_schema"]["type"] == "object"
        assert "properties" in o["input_schema"]
        assert "required" in o["input_schema"]


def test_les_descriptions_sont_substantielles():
    for o in OUTILS_SCHEMA:
        assert len(o["description"]) > 80, o["name"]


def test_le_system_prompt_impose_le_calcul_par_outil():
    assert "calculer_indemnite" in SYSTEM


Writing tests/test_outils.py


In [ ]:
%%writefile tests/test_boucle.py
"""Partie 2 - la boucle a la main. Tests FOURNIS, ne pas modifier."""
import pytest

from agent.boucle import executer, repondre
from tests.doubles import ClientBrutDouble, ClientBrutBoucle


# ── executer ──
def test_executer_route_correctement():
    r = executer("calculer_indemnite",
                 {"montant_dommages": 8500, "franchise": 500})
    assert r["indemnite"] == 8000.0


def test_outil_inconnu_message_explicite():
    with pytest.raises(ValueError) as e:
        executer("outil_fantome", {})
    assert "chercher_police" in str(e.value)


# ── la boucle ──
def test_boucle_complete():
    trace = []
    texte = repondre("Dossier POL-4471, refoulement, dommages 18000 $.",
                     ClientBrutDouble(), trace=trace)
    assert [n for n, _ in trace] == ["chercher_police",
                                     "rechercher_clause",
                                     "calculer_indemnite"]
    assert "17000" in texte


def test_aucune_erreur_sur_le_chemin_nominal():
    trace = []
    repondre("Dossier POL-4471, refoulement, dommages 18000 $.",
             ClientBrutDouble(), trace=trace)
    assert all(not err for _, err in trace)


def test_erreur_outil_capturee_sans_planter():
    trace = []
    texte = repondre("Dossier POL-9999, collision, dommages 4000 $.",
                     ClientBrutDouble(), trace=trace)
    assert trace[0] == ("chercher_police", True)
    assert "verifier" in texte.lower()


def test_garde_fou_max_tours():
    trace = []
    texte = repondre("boucle", ClientBrutBoucle(), max_tours=3, trace=trace)
    assert texte == "Nombre maximum d'etapes atteint."
    assert len(trace) == 3


def test_la_trace_reste_optionnelle():
    texte = repondre("Dossier POL-4471, refoulement, dommages 18000 $.",
                     ClientBrutDouble())
    assert "17000" in texte


Writing tests/test_boucle.py


In [ ]:
%%writefile tests/test_graphe.py
"""Partie 3 - le graphe LangGraph (version allegee, seance 1h30).

Tests FOURNIS. La memoire/checkpointing (threads) et l'evaluateur
de trajectoire sont retires de cette version - ce sont de bons
sujets pour une seance suivante, une fois la boucle de base solide.
"""
import pytest

pytest.importorskip("langgraph")

from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.graph import END, START, StateGraph
from langchain_core.messages import AIMessage

try:
    from agent.graphe import EtatAgent, OUTILS_LC, construire_agent
except ImportError:
    pytest.skip("agent/graphe.py pas encore ecrit (partie 3)",
                allow_module_level=True)

from tests.doubles import ModeleBoucle, ModeleDouble


def entree(question):
    return {"messages": [HumanMessage(content=question)],
            "dossier": None, "tours": 0}


# ── les outils emballes ──
def test_trois_outils_langchain():
    assert len(OUTILS_LC) == 3


def test_les_docstrings_deviennent_les_descriptions():
    par_nom = {o.name: o for o in OUTILS_LC}
    assert "POL-1234" in par_nom["chercher_police"].description
    assert "TOUJOURS" in par_nom["calculer_indemnite"].description


def test_un_outil_reste_appelable():
    par_nom = {o.name: o for o in OUTILS_LC}
    r = par_nom["calculer_indemnite"].invoke(
        {"montant_dommages": 8500, "franchise": 500})
    assert r["indemnite"] == 8000.0


# ── l'agent complet ──
def test_le_compteur_de_tours_avance():
    s = construire_agent(ModeleDouble()).invoke(
        entree("Dossier POL-4471, refoulement, dommages 18000 $."))
    assert s["tours"] >= 4


def test_sequence_des_outils():
    s = construire_agent(ModeleDouble()).invoke(
        entree("Dossier POL-4471, refoulement, dommages 18000 $."))
    outils = [m.name for m in s["messages"] if isinstance(m, ToolMessage)]
    assert outils == ["chercher_police", "rechercher_clause",
                      "calculer_indemnite"]


def test_reponse_finale_contient_le_calcul():
    s = construire_agent(ModeleDouble()).invoke(
        entree("Dossier POL-4471, refoulement, dommages 18000 $."))
    assert "17000" in s["messages"][-1].content


def test_erreur_outil_ne_fait_pas_planter():
    s = construire_agent(ModeleDouble()).invoke(
        entree("Dossier POL-9999, collision, dommages 4000 $."))
    tm = [m for m in s["messages"] if isinstance(m, ToolMessage)]
    assert len(tm) == 1
    assert "introuvable" in tm[0].content.lower()
    assert "verifier" in s["messages"][-1].content.lower()


def test_garde_fou_produit_un_message_propre():
    app = construire_agent(ModeleBoucle(), max_tours=4)
    s = app.invoke(entree("boucle"), {"recursion_limit": 100})
    assert "maximum" in s["messages"][-1].content.lower()
    assert s["tours"] <= 5


# ── demo optionnelle (non notee) ──
# Comprendre le reducteur add_messages sans dependre du code des
# etudiants - a lancer manuellement en classe si le temps le permet.
def _demo_add_messages_concatene():
    g = StateGraph(EtatAgent)
    g.add_node("n1", lambda e: {"messages": [AIMessage(content="a")]})
    g.add_node("n2", lambda e: {"messages": [AIMessage(content="b")]})
    g.add_edge(START, "n1")
    g.add_edge("n1", "n2")
    g.add_edge("n2", END)
    res = g.compile().invoke(entree("x"))
    assert len(res["messages"]) == 3


Writing tests/test_graphe.py


### Etat de depart

Les trois fichiers `agent/*.py`, tels que distribues (avec leurs `raise NotImplementedError`), doivent afficher `20 failed, 8 passed` — les 8 tests qui passent deja portent sur ce qui est fourni en exemple (`chercher_police`, `executer`, les outils LangChain deja emballes).

In [ ]:
%%writefile agent/outils.py
"""SQUELETTE - Les trois outils du portail sinistres.

Fonctions Python ORDINAIRES. Elles serviront a deux usages :
  - la boucle a la main, avec les schemas JSON de OUTILS_SCHEMA
  - le graphe LangGraph, via tool() dans agent/graphe.py

On ne les ecrit qu'une fois.

chercher_police() est donnee comme exemple complet : lisez-la
d'abord, elle montre le patron a suivre pour les deux autres.
"""
from __future__ import annotations

import re
from typing import Optional

BASE_DOSSIERS = {
    "POL-4471": {"garanties": ["collision", "vol", "vandalisme"],
                 "avenants": ["RE-04"],
                 "franchises": {"collision": 500, "refoulement": 1000,
                                "vol_pieces": 250, "vandalisme": 500}},
    "POL-8802": {"garanties": ["collision"], "avenants": [],
                 "franchises": {"collision": 500}},
}

CLAUSES = {
    "refoulement": ("Clause 7.3.2 - Les dommages par refoulement d'egout "
                    "sont couverts uniquement si l'avenant RE-04 a ete "
                    "souscrit. Franchise 1000 $. Maximum 25000 $. "
                    "(police-auto-2026, p.12)"),
    "collision": ("Clause 7.3.1 - Dommages par collision couverts. "
                  "Franchise 500 $. (police-auto-2026, p.12)"),
    "vol": ("Clause 9.2 - Vol de pieces couvert, franchise 250 $. "
            "Rapport de police obligatoire sous 48 h. "
            "(police-auto-2026, p.18)"),
    "escalade": ("Procedure - Toute reclamation depassant 50000 $ ou "
                 "impliquant des blessures est escaladee au superviseur. "
                 "(procedure-interne-2026, p.3)"),
}


# ── EXEMPLE COMPLET - lisez-la avant d'ecrire les deux autres ──────────
def chercher_police(numero: str) -> dict:
    """Recupere garanties, avenants et franchises d'une police
    d'assurance automobile a partir de son numero (format POL-1234).
    A utiliser des qu'un numero de police est mentionne."""
    if not re.fullmatch(r"POL-\d{4}", numero or ""):
        raise ValueError("Numero mal forme, format attendu: POL-1234")
    if numero not in BASE_DOSSIERS:
        raise KeyError(f"Police {numero} introuvable, verifiez le numero.")
    return BASE_DOSSIERS[numero]


# ── A COMPLETER (1) ─────────────────────────────────────────────────
def rechercher_clause(sujet: str) -> str:
    """Retrouve la clause applicable a un type de sinistre.
    Sujets connus : refoulement, collision, vol, escalade."""
    # Etapes :
    #   1. Nettoyer `sujet` : espaces autour retires, tout en minuscules.
    #   2. Si ce sujet nettoye n'est PAS une cle de CLAUSES :
    #      renvoyer une chaine qui commence par "Aucune clause" et qui
    #      liste les sujets connus, par ex avec list(CLAUSES).
    #      Ce n'est PAS une erreur : ne rien lever, juste renvoyer le texte.
    #   3. Sinon, renvoyer CLAUSES[sujet_nettoye] tel quel.
    raise NotImplementedError


# ── A COMPLETER (2) ─────────────────────────────────────────────────
def calculer_indemnite(montant_dommages: float, franchise: float,
                       maximum: Optional[float] = None) -> dict:
    """Calcule le montant net a verser. TOUJOURS utiliser cet outil
    pour tout calcul monetaire, ne jamais calculer de tete."""
    # Etapes :
    #   1. Si montant_dommages est None ou negatif : lever ValueError.
    #   2. Si franchise est None ou negative : lever ValueError.
    #   3. Calculer net = montant_dommages - franchise, jamais sous 0
    #      (utiliser max(0.0, ...)).
    #   4. Si un maximum est donne ET que net le depasse : net devient
    #      le maximum, et plafonne = True. Sinon plafonne = False.
    #   5. Renvoyer {"indemnite": net, "plafonne": plafonne}.
    raise NotImplementedError

# ── Schemas JSON pour l'API brute (bloc 1) ────────────────────────────
# Un dict par outil : name, description, input_schema.
# La description doit dire QUAND utiliser l'outil (et au besoin quand
# NE PAS l'utiliser) - c'est elle que Claude lit pour decider.
# Le premier est fait, sur le meme modele que chercher_police ci-dessus.
OUTILS_SCHEMA = [
    {
        "name": "chercher_police",
        "description": (
            "Recupere garanties, avenants et franchises d'une police "
            "d'assurance automobile a partir de son numero. A utiliser "
            "des qu'un numero de police (format POL-1234) est mentionne "
            "dans la question."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "numero": {"type": "string",
                           "description": "Format POL-1234"},
            },
            "required": ["numero"],
        },
    },
    # A COMPLETER (3) : le schema de rechercher_clause.
    #   - "sujet" est une string, avec une cle "enum" listant les
    #     sujets connus (list(CLAUSES)).
    # A COMPLETER (4) : le schema de calculer_indemnite.
    #   - trois proprietes : montant_dommages (number), franchise
    #     (number), maximum (number, PAS dans "required").
]

DISPATCH = {
    "chercher_police": chercher_police,
    "rechercher_clause": rechercher_clause,
    "calculer_indemnite": calculer_indemnite,
}

SYSTEM = (
    "Tu es un assistant d'analyse de sinistres pour OGI Assurance. "
    "Tu t'adresses a des courtiers internes.\n"
    "- Utilise les outils pour obtenir les faits. N'invente jamais une "
    "garantie ni un montant.\n"
    "- Pour tout calcul monetaire, utilise calculer_indemnite.\n"
    "- Cite toujours la clause et la page.\n"
    "- Si une information est absente, dis-le au lieu de supposer."
)


Writing agent/outils.py


In [ ]:
%%writefile agent/boucle.py
"""SQUELETTE - La boucle tool use ecrite a la main.

Aucune bibliotheque d'agent : uniquement le SDK anthropic.
C'est ce que LangGraph remplacera dans agent/graphe.py.
"""
from __future__ import annotations

from agent.outils import DISPATCH, OUTILS_SCHEMA, SYSTEM

MAX_TOURS = 5


# ── EXEMPLE COMPLET ─────────────────────────────────────────────────
def executer(nom: str, args: dict):
    """Route vers la bonne fonction Python."""
    fonction = DISPATCH.get(nom)
    if fonction is None:
        raise ValueError(f"Outil inconnu: {nom}. "
                          f"Disponibles: {list(DISPATCH)}")
    return fonction(**args)


# ── A COMPLETER ──────────────────────────────────────────────────────
def repondre(question: str, client, modele: str = "claude-haiku-4-5",
             max_tours: int = MAX_TOURS, trace: list | None = None) -> str:
    """Boucle complete. trace, si fourni, recoit (nom_outil, erreur)."""
    # Etape A - initialiser l'historique :
    #   messages = [{"role": "user", "content": question}]
    #
    # Etape B - repeter au plus max_tours fois :
    #
    #   B1. Appeler client.messages.create(model=modele, max_tokens=1024,
    #       system=SYSTEM, tools=OUTILS_SCHEMA, messages=messages).
    #       Le resultat a deux champs utiles : .stop_reason et .content
    #       (une liste de blocs, chacun avec .type == "text" ou "tool_use").
    #
    #   B2. Ajouter la reponse de Claude a l'historique :
    #       messages.append({"role": "assistant", "content": rep.content})
    #
    #   B3. Si rep.stop_reason != "tool_use" : Claude a fini, pas
    #       d'outil demande. Renvoyer le texte : concatener b.text pour
    #       chaque bloc b de rep.content ou b.type == "text".
    #
    #   B4. Sinon, pour CHAQUE bloc de rep.content ou b.type ==
    #       "tool_use" :
    #         - appeler executer(b.name, b.input) dans un try/except
    #         - si ca reussit : resultat = la valeur renvoyee, erreur=False
    #         - si ca leve une exception : resultat = message d'erreur
    #           lisible (str(exc)), erreur=True
    #         - si `trace` n'est pas None : trace.append((b.name, erreur))
    #         - construire un dict tool_result :
    #           {"type": "tool_result", "tool_use_id": b.id,
    #            "content": str(resultat), "is_error": erreur}
    #
    #   B5. Ajouter tous les tool_result de ce tour a l'historique en
    #       UN SEUL message : {"role": "user", "content": [...]}
    #
    # Etape C - si la boucle se termine sans reponse finale (max_tours
    #   atteint) : renvoyer "Nombre maximum d'etapes atteint."
    raise NotImplementedError


Writing agent/boucle.py


In [ ]:
%%writefile agent/graphe.py
"""SQUELETTE - Le meme agent, reconstruit en LangGraph.

Le graphe et le cablage (add_node / add_edge) sont deja ecrits en bas
du fichier. Ce qu'il reste a faire : emballer les deux outils
manquants, puis remplir le corps des trois fonctions-noeuds.
"""
from __future__ import annotations

from typing import Optional
from typing_extensions import TypedDict, Annotated

from langchain_core.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

from agent.outils import (
    chercher_police as _chercher_police,
    rechercher_clause as _rechercher_clause,
    calculer_indemnite as _calculer_indemnite,
)


# ── EXEMPLE COMPLET - le patron pour les deux outils suivants ─────────
@tool
def chercher_police(numero: str) -> dict:
    """Recupere garanties, avenants et franchises d'une police
    d'assurance automobile a partir de son numero (format POL-1234).
    A utiliser des qu'un numero de police est mentionne."""
    return _chercher_police(numero)


# ── A COMPLETER (1) - meme patron que ci-dessus ────────────────────
@tool
def rechercher_clause(sujet: str) -> str:
    """Retrouve la clause applicable a un type de sinistre.
    Sujets connus : refoulement, collision, vol, escalade."""
    raise NotImplementedError


# ── A COMPLETER (2) - meme patron que ci-dessus ────────────────────
@tool
def calculer_indemnite(montant_dommages: float, franchise: float,
                        maximum: Optional[float] = None) -> dict:
    """Calcule le montant net a verser. TOUJOURS utiliser cet outil
    pour tout calcul monetaire, ne jamais calculer de tete."""
    raise NotImplementedError


OUTILS_LC = [chercher_police, rechercher_clause, calculer_indemnite]
_PAR_NOM = {o.name: o for o in OUTILS_LC}


# ── Donne : l'etat partage entre les noeuds ────────────────────────
class EtatAgent(TypedDict):
    messages: Annotated[list, add_messages]
    dossier: str | None
    tours: int


def construire_agent(modele, max_tours: int = 5, checkpointer=None):
    lie = modele.bind_tools(OUTILS_LC)

    # ── A COMPLETER (3) ──────────────────────────────────────────
    def noeud_agent(etat):
        """Appelle le modele avec l'historique de messages, incremente
        `tours` de 1. Doit renvoyer un dict partiel de l'etat :
        {"messages": [reponse_du_modele], "tours": nouveau_tours}.
        Indice : lie.invoke(etat["messages"]) renvoie un AIMessage."""
        raise NotImplementedError

    # ── A COMPLETER (4) ──────────────────────────────────────────
    def noeud_outils(etat):
        """Execute chaque appel d'outil demande dans le DERNIER message
        (etat["messages"][-1].tool_calls), et renvoie
        {"messages": [ToolMessage(...), ...]}, un ToolMessage par appel.
        Chaque appel d'outil est un dict avec "name", "args", "id".
        Utiliser _PAR_NOM[appel["name"]].invoke(appel["args"]).
        En cas d'exception : le contenu du ToolMessage doit contenir le
        mot "verifiez" (en minuscule), pour rester coherent avec le
        message d'erreur de chercher_police. Ne jamais laisser
        l'exception remonter - le noeud ne doit jamais planter."""
        raise NotImplementedError

    # ── Donne ────────────────────────────────────────────────────
    def noeud_limite(etat):
        return {"messages": [AIMessage(
            content="Nombre maximum d'etapes atteint.")]}

    # ── A COMPLETER (5) ──────────────────────────────────────────
    def router(etat) -> str:
        """Choisit le prochain noeud :
          - "limite" si etat["tours"] a atteint ou depasse max_tours
          - "outils" si le dernier message est un AIMessage qui
            contient des tool_calls (liste non vide)
          - END sinon (Claude a repondu, on arrete)."""
        raise NotImplementedError

    # ── Donne : le cablage du graphe ─────────────────────────────
    g = StateGraph(EtatAgent)
    g.add_node("agent", noeud_agent)
    g.add_node("outils", noeud_outils)
    g.add_node("limite", noeud_limite)
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", router,
                             {"outils": "outils", "limite": "limite",
                              END: END})
    g.add_edge("outils", "agent")
    g.add_edge("limite", END)
    return g.compile(checkpointer=checkpointer)


Writing agent/graphe.py


In [ ]:
!pytest -q

F.FFFFF..FFFFFF..FFFFFFFF...                                             [100%]
=================================== FAILURES ===================================
_______________________ test_executer_route_correctement _______________________

    def test_executer_route_correctement():
>       r = executer("calculer_indemnite",
                     {"montant_dommages": 8500, "franchise": 500})

tests/test_boucle.py:10: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
agent/boucle.py:20: in executer
    return fonction(**args)
           ^^^^^^^^^^^^^^^^
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

montant_dommages = 8500, franchise = 500, maximum = None

    def calculer_indemnite(montant_dommages: float, franchise: float,
                           maximum: Optional[float] = None) -> dict:
        """Calcule le montant net a verser. TOUJOURS utiliser cet outil
        pour tout calcul monetaire, ne jamais calculer d

Sortie attendue : `20 failed, 8 passed`.

---
# Partie 1 — `agent/outils.py`

`chercher_police()` est fournie, entierement fonctionnelle — c'est le patron a suivre. `rechercher_clause()` et `calculer_indemnite()`, ainsi que le reste de `OUTILS_SCHEMA`, sont a ecrire.

### Etape 1.1 — `rechercher_clause(sujet)`

1. Nettoyer `sujet` : `.strip()` puis `.lower()`.
2. Verifier si ce sujet nettoye est une cle de `CLAUSES`.
3. Absent : renvoyer une chaine qui **commence par** `"Aucune clause"` et liste les sujets
   connus. Ce n'est pas une erreur — ne rien lever.
4. Present : renvoyer `CLAUSES[sujet_nettoye]` tel quel.

### Etape 1.2 — `calculer_indemnite(montant_dommages, franchise, maximum)`

1. `montant_dommages` `None` ou negatif -> `ValueError`.
2. `franchise` `None` ou negative -> `ValueError`.
3. `net = max(0.0, montant_dommages - franchise)`.
4. Si `maximum` est donne et que `net` le depasse : `net = maximum`, `plafonne = True`. Sinon
   `plafonne = False`.
5. Renvoyer `{"indemnite": net, "plafonne": plafonne}`.

### Etape 1.3 — completer `OUTILS_SCHEMA`

`chercher_police` est deja dans la liste. Ajouter :
- `rechercher_clause` : propriete `"sujet"` de type `string`, avec un `"enum"` = `list(CLAUSES)`.
- `calculer_indemnite` : trois proprietes (`montant_dommages`, `franchise` en `number` ;
  `maximum` en `number`), mais seules les deux premieres vont dans `"required"`.
- Chaque description fait plus de 80 caracteres et dit **quand** utiliser l'outil.


In [ ]:
%%writefile agent/outils.py
"""Les trois outils du portail sinistres.

Fonctions Python ORDINAIRES. Elles servent a deux usages :
  - la boucle a la main, avec les schemas JSON de OUTILS_SCHEMA
  - le graphe LangGraph, via tool() dans agent/graphe.py

On ne les ecrit qu'une fois.

chercher_police() est donnee comme exemple complet : lisez-la
d'abord, elle montre le patron a suivre pour les deux autres.
"""
from __future__ import annotations

import re
from typing import Optional

BASE_DOSSIERS = {
    "POL-4471": {"garanties": ["collision", "vol", "vandalisme"],
                 "avenants": ["RE-04"],
                 "franchises": {"collision": 500, "refoulement": 1000,
                                "vol_pieces": 250, "vandalisme": 500}},
    "POL-8802": {"garanties": ["collision"], "avenants": [],
                 "franchises": {"collision": 500}},
}

CLAUSES = {
    "refoulement": ("Clause 7.3.2 - Les dommages par refoulement d'egout "
                    "sont couverts uniquement si l'avenant RE-04 a ete "
                    "souscrit. Franchise 1000 $. Maximum 25000 $. "
                    "(police-auto-2026, p.12)"),
    "collision": ("Clause 7.3.1 - Dommages par collision couverts. "
                  "Franchise 500 $. (police-auto-2026, p.12)"),
    "vol": ("Clause 9.2 - Vol de pieces couvert, franchise 250 $. "
            "Rapport de police obligatoire sous 48 h. "
            "(police-auto-2026, p.18)"),
    "escalade": ("Procedure - Toute reclamation depassant 50000 $ ou "
                 "impliquant des blessures est escaladee au superviseur. "
                 "(procedure-interne-2026, p.3)"),
}


# ── EXEMPLE COMPLET - lisez-la avant d'ecrire les deux autres ──────────
def chercher_police(numero: str) -> dict:
    """Recupere garanties, avenants et franchises d'une police
    d'assurance automobile a partir de son numero (format POL-1234).
    A utiliser des qu'un numero de police est mentionne."""
    if not re.fullmatch(r"POL-\d{4}", numero or ""):
        raise ValueError("Numero mal forme, format attendu: POL-1234")
    if numero not in BASE_DOSSIERS:
        raise KeyError(f"Police {numero} introuvable, verifiez le numero.")
    return BASE_DOSSIERS[numero]


def rechercher_clause(sujet: str) -> str:
    """Retrouve la clause applicable a un type de sinistre.
    Sujets connus : refoulement, collision, vol, escalade."""
    sujet_nettoye = sujet.strip().lower()
    if sujet_nettoye not in CLAUSES:
        return ("Aucune clause trouvee pour ce sujet. Sujets connus : "
                + ", ".join(list(CLAUSES)))
    return CLAUSES[sujet_nettoye]


def calculer_indemnite(montant_dommages: float, franchise: float,
                       maximum: Optional[float] = None) -> dict:
    """Calcule le montant net a verser. TOUJOURS utiliser cet outil
    pour tout calcul monetaire, ne jamais calculer de tete."""
    if montant_dommages is None or montant_dommages < 0:
        raise ValueError("montant_dommages doit etre un nombre positif.")
    if franchise is None or franchise < 0:
        raise ValueError("franchise doit etre un nombre positif.")

    net = max(0.0, montant_dommages - franchise)
    plafonne = maximum is not None and net > maximum
    if plafonne:
        net = maximum

    return {"indemnite": net, "plafonne": plafonne}

# ── Schemas JSON pour l'API brute (bloc 1) ────────────────────────────
# Un dict par outil : name, description, input_schema.
# La description doit dire QUAND utiliser l'outil (et au besoin quand
# NE PAS l'utiliser) - c'est elle que Claude lit pour decider.
# Le premier est fait, sur le meme modele que chercher_police ci-dessus.
OUTILS_SCHEMA = [
    {
        "name": "chercher_police",
        "description": (
            "Recupere garanties, avenants et franchises d'une police "
            "d'assurance automobile a partir de son numero. A utiliser "
            "des qu'un numero de police (format POL-1234) est mentionne "
            "dans la question."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "numero": {"type": "string",
                           "description": "Format POL-1234"},
            },
            "required": ["numero"],
        },
    },
    {
        "name": "rechercher_clause",
        "description": (
            "Retrouve le texte de la clause applicable a un type de "
            "sinistre, avec sa reference et sa page. A utiliser avant de "
            "citer une couverture, une franchise ou un plafond : ne "
            "jamais reciter une clause de memoire."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "sujet": {
                    "type": "string",
                    "description": "Type de sinistre concerne.",
                    "enum": list(CLAUSES),
                },
            },
            "required": ["sujet"],
        },
    },
    {
        "name": "calculer_indemnite",
        "description": (
            "Calcule le montant net a verser une fois la franchise "
            "appliquee, et indique si le plafond de la garantie a ete "
            "atteint. TOUJOURS utiliser cet outil pour tout calcul "
            "monetaire, ne jamais calculer un montant de tete."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "montant_dommages": {
                    "type": "number",
                    "description": "Montant total des dommages estimes, en dollars.",
                },
                "franchise": {
                    "type": "number",
                    "description": "Franchise applicable, en dollars.",
                },
                "maximum": {
                    "type": "number",
                    "description": "Plafond de la garantie, en dollars, si applicable.",
                },
            },
            "required": ["montant_dommages", "franchise"],
        },
    },
]

DISPATCH = {
    "chercher_police": chercher_police,
    "rechercher_clause": rechercher_clause,
    "calculer_indemnite": calculer_indemnite,
}

SYSTEM = (
    "Tu es un assistant d'analyse de sinistres pour OGI Assurance. "
    "Tu t'adresses a des courtiers internes.\n"
    "- Utilise les outils pour obtenir les faits. N'invente jamais une "
    "garantie ni un montant.\n"
    "- Pour tout calcul monetaire, utilise calculer_indemnite.\n"
    "- Cite toujours la clause et la page.\n"
    "- Si une information est absente, dis-le au lieu de supposer."
)


Overwriting agent/outils.py


In [ ]:
!pytest -q tests/test_outils.py

.............                                                            [100%]
13 passed in 0.03s


Sortie attendue : `13 passed`.

---
# Partie 2 — `agent/boucle.py` (Tool Use)

Ici, pas de framework : uniquement le SDK Anthropic. C'est le "tool use" tel quel.

`executer()` est fournie : un simple aiguillage vers `DISPATCH`. Il reste `repondre()`, la
boucle elle-meme, a ecrire.

### Etape 2.1 — initialiser

```python
messages = [{"role": "user", "content": question}]
```

### Etape 2.2 — la boucle (au plus `max_tours` fois)

1. `rep = client.messages.create(model=modele, max_tokens=1024, system=SYSTEM, tools=OUTILS_SCHEMA, messages=messages)`.
2. Ajouter la reponse a l'historique : `messages.append({"role": "assistant", "content": rep.content})`.
3. Si `rep.stop_reason != "tool_use"` : Claude a fini — renvoyer le texte (concatener `b.text`
   pour chaque bloc `b` de `rep.content` ou `b.type == "text"`).
4. Sinon, pour **chaque** bloc `b.type == "tool_use"` : appeler `executer(b.name, b.input)` dans
   un `try`/`except`. Succes -> `resultat` = valeur renvoyee, `erreur = False`. Exception ->
   `resultat = str(exception)`, `erreur = True`.
5. Si `trace` n'est pas `None` : `trace.append((b.name, erreur))`.
6. Construire un dict par outil execute : `{"type": "tool_result", "tool_use_id": b.id, "content": str(resultat), "is_error": erreur}`.
7. Ajouter tous les `tool_result` de ce tour en **un seul** message : `{"role": "user", "content": [...]}`.

### Etape 2.3 — le garde-fou

Si la boucle atteint `max_tours` sans reponse finale : renvoyer `"Nombre maximum d'etapes atteint."`


In [ ]:
%%writefile agent/boucle.py
"""La boucle tool use ecrite a la main.

Aucune bibliotheque d'agent : uniquement le SDK anthropic.
C'est ce que LangGraph remplacera dans agent/graphe.py.
"""
from __future__ import annotations

from agent.outils import DISPATCH, OUTILS_SCHEMA, SYSTEM

MAX_TOURS = 5


# ── EXEMPLE COMPLET ─────────────────────────────────────────────────
def executer(nom: str, args: dict):
    """Route vers la bonne fonction Python."""
    fonction = DISPATCH.get(nom)
    if fonction is None:
        raise ValueError(f"Outil inconnu: {nom}. "
                          f"Disponibles: {list(DISPATCH)}")
    return fonction(**args)


def repondre(question: str, client, modele: str = "claude-haiku-4-5",
             max_tours: int = MAX_TOURS, trace: list | None = None) -> str:
    """Boucle complete. trace, si fourni, recoit (nom_outil, erreur)."""
    messages = [{"role": "user", "content": question}]

    for _ in range(max_tours):
        # demande a Claude la prochaine action (texte final ou appel d'outil)
        rep = client.messages.create(
            model=modele, max_tokens=1024, system=SYSTEM,
            tools=OUTILS_SCHEMA, messages=messages)

        # la reponse de Claude rejoint l'historique telle quelle
        messages.append({"role": "assistant", "content": rep.content})

        # pas d'outil demande : Claude a fini, on renvoie son texte
        if rep.stop_reason != "tool_use":
            return "".join(b.text for b in rep.content
                           if b.type == "text")

        # sinon, executer chaque outil demande dans ce tour
        resultats = []
        for b in rep.content:
            if b.type != "tool_use":
                continue
            try:
                resultat, erreur = executer(b.name, b.input), False
            except Exception as exc:
                resultat, erreur = str(exc), True

            if trace is not None:
                trace.append((b.name, erreur))

            resultats.append({"type": "tool_result",
                              "tool_use_id": b.id,
                              "content": str(resultat),
                              "is_error": erreur})

        # tous les resultats de ce tour repartent en UN SEUL message,
        # en role "user" (c'est la regle de l'API, meme si contre-intuitif)
        messages.append({"role": "user", "content": resultats})

    # la boucle s'est terminee sans reponse finale : le garde-fou
    return "Nombre maximum d'etapes atteint."


Overwriting agent/boucle.py


In [ ]:
!pytest -q tests/test_outils.py tests/test_boucle.py

....................                                                     [100%]
20 passed in 0.10s


Sortie attendue : `20 passed` (`test_outils.py` + `test_boucle.py`).

`agent/graphe.py` existe deja sur disque, mais ses noeuds ne sont pas encore ecrits. On filtre donc ici sur les deux premiers fichiers de tests pour rester sur le perimetre de la partie 2. Un `pytest -q` sans filtre afficherait `22 passed, 6 failed` a ce stade.


### Essai contre le vrai Claude *(Appel API reel — utilise Haiku, quelques centimes)*

Optionnel : observer le comportement reel de Claude sur le meme scenario que les tests.


In [ ]:
import anthropic

assert ANTHROPIC_API_KEY, "Ajoutez ANTHROPIC_API_KEY dans les secrets Colab avant cette cellule."

from agent.boucle import repondre

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

trace_reelle = []
reponse_reelle = repondre(
    "Dossier POL-4471, refoulement d'egout au sous-sol, "
    "dommages 18 000 $. Est-ce couvert et combien ?",
    client, trace=trace_reelle)

print(reponse_reelle)
print()
print("Trace des outils appeles :", trace_reelle)


**Montant à verser : 17 000 $**

- Dommages déclarés : 18 000 $
- Franchise applicable : 1 000 $
- Plafond de garantie : 25 000 $ (non atteint)
- **Indemnité nette : 17 000 $**

Trace des outils appeles : [('chercher_police', False), ('rechercher_clause', False), ('calculer_indemnite', False)]


---
# Partie 3 — `agent/graphe.py` (Agent)

Ici, LangGraph joue le role du framework d'agent : il gere la boucle a notre place.

Sont fournis : l'etat `EtatAgent`, l'outil `chercher_police` emballe avec `@tool`, le
noeud `limite`, et tout le cablage du graphe (`add_node` / `add_edge`) en bas du fichier. Il
reste 5 blancs a ecrire, marques `A COMPLETER (1)` a `(5)`.

### Etape 3.1 — emballer les deux outils manquants (1) et (2)

Meme patron que `chercher_police` :

```python
@tool
def rechercher_clause(sujet: str) -> str:
    # reprendre exactement la docstring de la fonction originale
    return _rechercher_clause(sujet)
```

La docstring devient automatiquement la description que Claude lira — elle doit donc rester
identique a celle de `agent/outils.py`.

### Etape 3.2 — `noeud_agent` (3)

1. `rep = lie.invoke(etat["messages"])` (`lie = modele.bind_tools(OUTILS_LC)`, deja defini).
2. Renvoyer `{"messages": [rep], "tours": etat.get("tours", 0) + 1}`.

### Etape 3.3 — `noeud_outils` (4)

1. `dernier = etat["messages"][-1]`.
2. Pour chaque `appel` dans `dernier.tool_calls` (`dict` avec `"name"`, `"args"`, `"id"`) :
   `_PAR_NOM[appel["name"]].invoke(appel["args"])`.
3. `try`/`except` autour de l'appel. En cas d'exception, le contenu du message doit contenir
   `"verifiez"` — coherent avec le message d'erreur de `chercher_police`.
4. Construire un `ToolMessage(content=..., tool_call_id=appel["id"], name=appel["name"])` par
   appel, renvoyer `{"messages": [...]}`.

### Etape 3.4 — `router` (5)

1. `etat["tours"]` a atteint ou depasse `max_tours` -> `"limite"`.
2. Dernier message = `AIMessage` avec `tool_calls` non vide -> `"outils"`.
3. Sinon -> `END`.


In [ ]:
%%writefile agent/graphe.py
"""Le meme agent, reconstruit en LangGraph.

Le graphe et le cablage (add_node / add_edge) sont deja ecrits en bas
du fichier. Ce qu'il reste a faire : emballer les deux outils
manquants, puis remplir le corps des trois fonctions-noeuds.
"""
from __future__ import annotations

from typing import Optional
from typing_extensions import TypedDict, Annotated

from langchain_core.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

from agent.outils import (
    chercher_police as _chercher_police,
    rechercher_clause as _rechercher_clause,
    calculer_indemnite as _calculer_indemnite,
)


# ── EXEMPLE COMPLET - le patron pour les deux outils suivants ─────────
@tool
def chercher_police(numero: str) -> dict:
    """Recupere garanties, avenants et franchises d'une police
    d'assurance automobile a partir de son numero (format POL-1234).
    A utiliser des qu'un numero de police est mentionne."""
    return _chercher_police(numero)


@tool
def rechercher_clause(sujet: str) -> str:
    """Retrouve la clause applicable a un type de sinistre.
    Sujets connus : refoulement, collision, vol, escalade."""
    return _rechercher_clause(sujet)


@tool
def calculer_indemnite(montant_dommages: float, franchise: float,
                        maximum: Optional[float] = None) -> dict:
    """Calcule le montant net a verser. TOUJOURS utiliser cet outil
    pour tout calcul monetaire, ne jamais calculer de tete."""
    return _calculer_indemnite(montant_dommages, franchise, maximum)


OUTILS_LC = [chercher_police, rechercher_clause, calculer_indemnite]
_PAR_NOM = {o.name: o for o in OUTILS_LC}


# ── Donne : l'etat partage entre les noeuds ────────────────────────
class EtatAgent(TypedDict):
    messages: Annotated[list, add_messages]
    dossier: str | None
    tours: int


def construire_agent(modele, max_tours: int = 5, checkpointer=None):
    lie = modele.bind_tools(OUTILS_LC)

    def noeud_agent(etat):
        """Appelle le modele avec l'historique de messages et incremente
        le compteur de tours."""
        rep = lie.invoke(etat["messages"])
        return {"messages": [rep], "tours": etat.get("tours", 0) + 1}

    def noeud_outils(etat):
        """Execute chaque appel d'outil demande par le dernier message,
        et renvoie un ToolMessage par appel. Une exception est capturee
        et renvoyee comme contenu du message plutot que de faire planter
        le noeud (chercher_police() y ajoute deja le mot "verifiez")."""
        dernier = etat["messages"][-1]
        resultats = []
        for appel in dernier.tool_calls:
            outil = _PAR_NOM[appel["name"]]
            try:
                sortie = outil.invoke(appel["args"])
            except Exception as exc:
                sortie = str(exc)
            resultats.append(ToolMessage(content=str(sortie),
                                         tool_call_id=appel["id"],
                                         name=appel["name"]))
        return {"messages": resultats}

    # ── Donne ────────────────────────────────────────────────────
    def noeud_limite(etat):
        return {"messages": [AIMessage(
            content="Nombre maximum d'etapes atteint.")]}

    def router(etat) -> str:
        """Decide du prochain noeud : le garde-fou l'emporte, sinon on
        route vers les outils si Claude en a demande, sinon on arrete."""
        if etat.get("tours", 0) >= max_tours:
            return "limite"
        dernier = etat["messages"][-1]
        if getattr(dernier, "tool_calls", None):
            return "outils"
        return END

    # ── Donne : le cablage du graphe ─────────────────────────────
    g = StateGraph(EtatAgent)
    g.add_node("agent", noeud_agent)
    g.add_node("outils", noeud_outils)
    g.add_node("limite", noeud_limite)
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", router,
                             {"outils": "outils", "limite": "limite",
                              END: END})
    g.add_edge("outils", "agent")
    g.add_edge("limite", END)
    return g.compile(checkpointer=checkpointer)


Overwriting agent/graphe.py


In [ ]:
!pytest -q

............................                                             [100%]
28 passed in 0.40s


Sortie attendue : `28 passed`.

### Detail par fichier

In [ ]:
!pytest -v --tb=short

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/portail-agent
configfile: pytest.ini
testpaths: tests
plugins: langsmith-0.11.0, anyio-4.14.2, typeguard-4.6.0
collected 28 items                                                             

tests/test_boucle.py::test_executer_route_correctement PASSED            [  3%]
tests/test_boucle.py::test_outil_inconnu_message_explicite PASSED        [  7%]
tests/test_boucle.py::test_boucle_complete PASSED                        [ 10%]
tests/test_boucle.py::test_aucune_erreur_sur_le_chemin_nominal PASSED    [ 14%]
tests/test_boucle.py::test_erreur_outil_capturee_sans_planter PASSED     [ 17%]
tests/test_boucle.py::test_garde_fou_max_tours PASSED                    [ 21%]
tests/test_boucle.py::test_la_trace_reste_optionnelle PASSED             [ 25%]
tests/test_graphe.py::test_trois_outils_lang

---
# Synthese

**Partie 1 -- les outils.** `chercher_police` etant fournie comme patron, `rechercher_clause` et
`calculer_indemnite` suivent la meme structure : valider les entrees, chercher dans un
dictionnaire, renvoyer un resultat simple. Le point le plus facile a rater est
`test_indemnite_jamais_negative` : sans `max(0.0, ...)`, un montant de dommages inferieur a la
franchise produirait une indemnite negative, ce qui n'a pas de sens pour un versement.

**Partie 2 -- la boucle a la main.** La difficulte n'est pas algorithmique mais dans le respect
strict du contrat de l'API : le `tool_use_id` doit faire l'aller-retour exact entre la demande et
le resultat, et le resultat repart en role `"user"` (pas `"assistant"`), ce qui n'est pas
intuitif au premier abord. Le garde-fou `max_tours` a ete verifie contre `ClientBrutBoucle`, qui
redemande volontairement le meme outil a l'infini -- sans lui, cette boucle ne s'arreterait jamais.

**Partie 3 -- LangGraph.** Le plus instructif ici est de voir le meme comportement se redecouper
en noeuds independants : `noeud_agent` ne fait qu'appeler le modele, `noeud_outils` ne fait
qu'executer les outils, `router` ne fait que decider de la suite. Aucune de ces responsabilites
ne se chevauche, ce qui rend chaque noeud testable isolement -- contrairement a la boucle a la
main ou tout est imbrique dans une seule fonction.

**Defi commun aux parties 2 et 3.** La gestion d'erreur ne doit jamais faire planter l'appelant.
Dans les deux cas, une exception levee par un outil (police introuvable, par exemple) est
capturee et transformee en un message que Claude peut lire et interpreter -- c'est ce qui permet
au systeme de repondre "verifiez le numero" au lieu de crasher.
